# 🤖 TechMart RAG Chatbot: Complete End-to-End Pipeline
### Built with LangChain, ChromaDB, and Google Gemini

This notebook walks through the complete Retrieval-Augmented Generation (RAG) system for **TechMart Online Store**:
1. **Document Loading**: Ingesting `policy.txt` and `products.xlsx`.
2. **Chunking & Semantic Transformation**: Splitting policy sections cleanly and transforming tabular product rows into rich natural language documents.
3. **Embedding & Vector Indexing**: Creating vector embeddings with Gemini (`models/gemini-embedding-2`) and persisting them in **ChromaDB**.
4. **Retriever Setup**: Configuring similarity search with metadata attribution.
5. **Gemini Chatbot (Charlie)**: Persona-aligned system prompt and conversational QA chain.
6. **Interactive Chat Queries**: Real-world tests for product inquiries, stock checks, and policy questions.

In [ ]:
# Cell 1: Install Required Packages
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-google-genai langchain-chroma chromadb pandas openpyxl python-dotenv

In [ ]:
# Cell 2: Imports & Environment Setup
import os
import pandas as pd
from dotenv import load_dotenv

# LangChain & Vector Store Imports
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

# Set your Google Gemini API Key:
# Get a free key at: https://aistudio.google.com/app/apikey
if "GOOGLE_API_KEY" not in os.environ:
    # Prompt for key if not set
    import getpass
    api_key = getpass.getpass("Enter your Google Gemini API Key: ")
    os.environ["GOOGLE_API_KEY"] = api_key

print("✓ Environment & API Key ready.")

## 1. Document Loading
Load unstructured text from `policy.txt` and tabular product data from `products.xlsx`.

In [ ]:
# Cell 3: Load Policy Text and Products Excel
policy_loader = TextLoader("policy.txt", encoding="utf-8")
policy_docs = policy_loader.load()
print(f"Loaded {len(policy_docs)} policy document(s).")

df_products = pd.read_excel("products.xlsx")
print(f"Loaded {len(df_products)} products from products.xlsx.")
print("\n--- Sample Products ---")
display(df_products.head(3))

## 2. Chunking & Semantic Transformation
- **`policy.txt`**: Section-level recursive character chunking.
- **`products.xlsx`**: Row-to-document transformation with complete labels (`Brand`, `Model Name`, `Category`, `Price`, `Stock`, `Key Specifications`, `Best For`) and metadata.

In [ ]:
# Cell 4: Chunk Policy Text
policy_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " "]
)
policy_chunks = policy_splitter.split_documents(policy_docs)

for i, chunk in enumerate(policy_chunks):
    first_line = chunk.page_content.strip().split("\n")[0]
    chunk.metadata.update({
        "source": "policy.txt",
        "doc_type": "policy",
        "section": first_line[:40],
        "chunk_id": i
    })

print(f"Policy divided into {len(policy_chunks)} cohesive chunks.")
print("Sample Chunk:")
print(policy_chunks[1].page_content)

In [ ]:
# Cell 5: Transform Excel Rows into Semantic Documents
product_docs = []

for index, row in df_products.iterrows():
    brand = str(row.get('Brand', '')).strip()
    model = str(row.get('Model Name', '')).strip()
    category = str(row.get('Category', '')).strip()
    specs = str(row.get('Key Specifications', '')).strip()
    best_for = str(row.get('Best For', '')).strip()
    price = float(row.get('price', 0.0))
    stock = int(row.get('stock', 0)) if pd.notna(row.get('stock')) else 0
    stock_status = f"{stock} units in stock" if stock > 0 else "Currently out of stock"

    content = (
        f"Product: {brand} {model}\n"
        f"Category: {category}\n"
        f"Price: ${price:.2f}\n"
        f"Availability: {stock_status}\n"
        f"Key Specifications: {specs}\n"
        f"Best For: {best_for}"
    )

    doc = Document(
        page_content=content,
        metadata={
            "source": "products.xlsx",
            "doc_type": "product",
            "category": category,
            "brand": brand,
            "model": model,
            "price": price,
            "stock": stock,
            "in_stock": stock > 0
        }
    )
    product_docs.append(doc)

print(f"Transformed {len(product_docs)} product rows into semantic documents.")
print("Sample Product Document:")
print(product_docs[0].page_content)

## 3. Embedding & ChromaDB Indexing
We combine all chunks and index them into **ChromaDB** using Google Gemini Embeddings (`models/gemini-embedding-2`).

In [ ]:
# Cell 6: Build Chroma Vector Store
all_documents = policy_chunks + product_docs
print(f"Total items to index: {len(all_documents)}")

# Initialize Gemini Embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

# Create and persist Chroma database
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    collection_name="techmart_knowledge_base",
    persist_directory="./chroma_db"
)

print("✓ ChromaDB indexed and persisted to ./chroma_db")

## 4. Retriever Verification
Test similarity search for both policy clauses and product recommendations.

In [ ]:
# Cell 7: Verify Retrieval
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("--- Test 1: Policy Retrieval ---")
query_1 = "What is the return policy for electronics like phones and laptops?"
for doc in retriever.invoke(query_1):
    print(f"[{doc.metadata.get('doc_type').upper()} | {doc.metadata.get('source')}]:")
    print(doc.page_content[:150] + "...")

print("\n--- Test 2: Product Retrieval ---")
query_2 = "Lightweight laptop for students"
for doc in retriever.invoke(query_2):
    print(f"[{doc.metadata.get('doc_type').upper()} | {doc.metadata.get('source')}]:")
    print(doc.page_content[:150] + "...")

## 5. Gemini Chatbot: Persona & RAG Pipeline
Configure **Charlie**, TechMart's friendly AI sales and support specialist, with prompt grounding and multi-turn context support.

In [ ]:
# Cell 8: Initialize Gemini LLM & Chatbot Pipeline
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0.3,
    google_api_key=os.environ["GOOGLE_API_KEY"]
)

SYSTEM_PROMPT = """You are Charlie, an expert, courteous, and highly knowledgeable AI sales and customer support assistant at TechMart Online Store.

Your mission:
1. Help customers find products, compare specs, check real-time stock and prices.
2. Clarify store policies on returns, refunds, shipping options, warranties, and support.

Rules:
- Ground answers strictly in the provided Context.
- State prices ($) and stock availability clearly.
- If an item is out of stock, mention it proactively.
- For policy questions, cite exact timeframes (e.g., 30-day return, 15-day electronics, 5 business days for refunds).
- Never hallucinate unlisted items or policies. If not found, politely let the customer know.

Context:
{context}
"""

def ask_charlie(question: str, history: list = None, k: int = 4):
    docs = retriever.invoke(question)
    context_text = "\n\n---\n\n".join([
        f"[{doc.metadata.get('doc_type', 'doc').upper()} | Source: {doc.metadata.get('source')}]\n{doc.page_content}"
        for doc in docs
    ])
    
    messages = [
        HumanMessage(content=f"[SYSTEM INSTRUCTION]:\n{SYSTEM_PROMPT.format(context=context_text)}")
    ]
    
    if history:
        for role, text in history:
            if role == 'user':
                messages.append(HumanMessage(content=text))
            else:
                messages.append(AIMessage(content=text))
                
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    raw_content = response.content
    if isinstance(raw_content, list):
        text_parts = [p.get('text', '') if isinstance(p, dict) else str(p) for p in raw_content]
        answer_text = '\n'.join(text_parts)
    else:
        answer_text = str(raw_content)
    return answer_text, docs

print("✓ Charlie RAG Chatbot is online!")

## 6. Sample Interactive Conversations
Test multi-turn and domain-specific questions.

In [ ]:
# Cell 9: Test Questions
queries = [
    "Can I return a laptop after 20 days?",
    "What is the best laptop you have for students, and is it in stock?",
    "How much does express shipping cost, and how fast is it?"
]

for q in queries:
    print(f"\n🧑 User: {q}")
    answer, sources = ask_charlie(q)
    print(f"🤖 Charlie: {answer}")
    print(f"   [Retrieved {len(sources)} sources from ChromaDB]")